# Sensitivity Analysis Notebook
This notebook evaluates how mortality pricing reacts to parameters such as interest rates and mortality rate shocks, using configuration defaults from `config.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import config

df = pd.read_csv('../data/morality_table.csv')

def get_premium(age, gender, term, sum_assured, interest_rate, shock=1.0):
    i = interest_rate / 100.0
    v = 1 / (1 + i)
    base_rates = df['qx'].values
    gender_factor = config.GENDER_FACTORS[gender]
    rates = np.clip(base_rates * gender_factor * shock, 0, 1)
    
    tpx = np.zeros(len(df) - age)
    tpx[0] = 1.0
    for t in range(1, len(tpx)):
        tpx[t] = tpx[t-1] * (1.0 - rates[age + t - 1])
        
    n = min(term, len(tpx) - 1)
    nsp = sum((v ** (t + 1)) * tpx[t] * rates[age + t] for t in range(n))
    a_due = sum((v ** t) * tpx[t] for t in range(n))
    
    return (nsp * sum_assured) / a_due if a_due > 0 else nsp * sum_assured

In [ ]:
# Interest Rate Sensitivity
rates = np.linspace(1.0, 12.0, 15)
premiums = [get_premium(config.DEFAULT_AGE, 'Male', config.DEFAULT_TERM, config.SUM_ASSURED, r) for r in rates]

plt.figure(figsize=(10, 5))
plt.plot(rates, premiums, 'o-', color='purple')
plt.title('Premium Sensitivity to Discount / Interest Rates')
plt.xlabel('Interest Rate (%)')
plt.ylabel('Annual Premium ($)')
plt.grid(True)
plt.show()

In [ ]:
# Mortality Shock Sensitivity
shocks = np.linspace(0.8, 1.5, 8)
shk_premiums = [get_premium(config.DEFAULT_AGE, 'Male', config.DEFAULT_TERM, config.SUM_ASSURED, config.INTEREST_RATE * 100, shock=s) for s in shocks]

plt.figure(figsize=(10, 5))
plt.plot(shocks, shk_premiums, 's-', color='red')
plt.title('Premium Sensitivity to Mortality Shocks (Multiplier)')
plt.xlabel('Mortality Multiplier')
plt.ylabel('Annual Premium ($)')
plt.grid(True)
plt.show()